# Day 19 — Grouping, Aggregation & Split-Apply-Combine Workflows

This notebook covers Split-Apply-Combine Framework, Named Aggregations, Group-level Transformations (`.transform()`), and Group Filtering (`.filter()`)

## Validation goals
- Understand the Split-Apply-Combine paradigm and manage lazy `DataFrameGroupBy` objects[cite: 16]
- Execute multi-column groupings using `as_index=False` to preserve grouping keys as standard DataFrame columns[cite: 16]
- Apply named aggregations (`avg_revenue=('revenue', 'mean')`) and multi-metric dictionary mappings using `.agg()`[cite: 16]
- Contrast aggregation (`.agg()`) with shape-preserving group transformations (`.transform()`) for feature engineering[cite: 16]
- Implement conditional group-level filtering using `.filter()` and evaluate C-level method performance vs. Python lambdas[cite: 16]

Analyzing aggregate metrics across categorical segments is a fundamental requirement in data analytics and machine learning feature engineering. Whether calculating total revenue per store, average latency per service endpoint, or maximum temperature per region, data summarization relies on the **Split-Apply-Combine** pattern.

Pandas implements this pattern through the `.groupby()` engine, providing optimized C-level performance for grouping, aggregating, transforming, and filtering tabular datasets.

### Core Concepts & Deep Dive

1. The Split-Apply-Combine Paradigm
Introduced by Hadley Wickham, this operational framework defines how grouped data is processed:

- Split: The dataset is partitioned into separate groups based on one or more keys (column values, index levels, or custom mapping functions).

- Apply: A function (aggregation, transformation, or filtering) is computed independently on each individual group in parallel or sequence.

- Combine: The results of the group-wise computations are concatenated and reindexed into a unified output DataFrame or Series.

2. Grouping Mechanics & Indexing (as_index)
- Groupby Objects: Calling `df.groupby('column')` returns a `DataFrameGroupBy` object. It does not compute aggregations immediately (lazy evaluation); it merely computes group boundary pointers.

- The Index Dilemma: By default, Pandas sets the grouping keys as the resulting DataFrame's hierarchical index (`MultiIndex`). While useful for nested indexing, setting `as_index=False` retains the grouping keys as standard columns, which is often preferred for downstream machine learning pipelines and database insertion

3. Advanced Aggregations (`.agg()`)
Rather than calling basic reduction methods like `.mean()` or `.sum()` directly on a groupby object, the `.agg()` (or `.aggregate()`) method unlocks advanced capabilities:

- Multi-Function Mapping: Applying multiple statistical metrics simultaneously to columns (e.g., passing a list of functions like `['mean', 'std', 'max']`).

- Named Aggregations: Modern Pandas allows defining clean, explicit output column names directly within `.agg()` using keyword arguments (e.g., `avg_price=('price', 'mean')`, `total_qty=('quantity', 'sum')`).

4. Aggregation vs. Transformation (`.transform()`)
- Aggregation (.agg()):  Reduces the dimensionality of the data, returning one scalar value per group (e.g., calculating the average salary per department, returning one row per department).
- Transformation (`.transform()`): Applies a function to each group but retains the original DataFrame's shape, broadcasting group-level metrics back to every individual row. This is essential for feature engineering, such as calculating a row's deviation from its group mean ($x_i - \bar{x}_{\text{group}}$).

### Practical Implementation: Grouping & Aggregation Pipeline

Here is a comprehensive script demonstrating multi-column grouping, named aggregations, group-wise filtering, and data transformation:

In [1]:
import numpy as np
import pandas as pd

# Setup mock transactional dataset
np.random.seed(42)
n_rows = 1000

df = pd.DataFrame({
    "store_id": np.random.choice(["STORE_A", "STORE_B", "STORE_C"], size=n_rows),
    "category": np.random.choice(["Electronics", "Apparel", "Groceries"], size=n_rows),
    "units_sold": np.random.randint(1, 20, size=n_rows),
    "revenue": np.round(np.random.uniform(20.0, 500.0, size=n_rows), 2)
})

# ==========================================
# 1. Basic Groupby & Named Aggregations
# ==========================================
# Group by store and category, calculating custom named aggregate metrics
summary_df = df.groupby(["store_id", "category"], as_index=False).agg(
    total_transactions=("revenue", "count"),
    total_revenue=("revenue", "sum"),
    mean_units_sold=("units_sold", "mean"),
    max_revenue=("revenue", "max")
)

print("--- Multi-Column Groupby with Named Aggregations ---")
print(summary_df.head(6))


# ==========================================
# 2. Multiple Functions on Specific Columns
# ==========================================
# Apply different statistical functions across distinct columns using a dictionary
multi_stat_df = df.groupby("store_id").agg({
    "revenue": ["min", "median", "max"],
    "units_sold": ["sum", "mean"]
})

print("\n--- Multi-Function Dictionary Aggregation ---")
print(multi_stat_df)


# ==========================================
# 3. Group Transformation (Broadcasting Back to Original Shape)
# ==========================================
# Calculate the mean revenue per category, then compute each transaction's deviation from its category mean
df["category_mean_revenue"] = df.groupby("category")["revenue"].transform("mean")
df["revenue_delta_from_mean"] = df["revenue"] - df["category_mean_revenue"]

print("\n--- Group Transformation (Shape-Preserving Broadcast) ---")
print(df[["store_id", "category", "revenue", "category_mean_revenue", "revenue_delta_from_mean"]].head(3))


# ==========================================
# 4. Group Filtering (.filter)
# ==========================================
# Filter out groups where the total category revenue is below a certain threshold
high_performing_categories = df.groupby("category").filter(lambda x: x["revenue"].sum() > 150000.0)
print(f"\n--- Filtered Groups (Original rows: {len(df)}, Filtered rows: {len(high_performing_categories)}) ---")

--- Multi-Column Groupby with Named Aggregations ---
  store_id     category  total_transactions  total_revenue  mean_units_sold  \
0  STORE_A      Apparel                 125       31604.70         9.512000   
1  STORE_A  Electronics                 118       28925.71         9.576271   
2  STORE_A    Groceries                 112       31278.26         9.848214   
3  STORE_B      Apparel                 102       28591.62        10.156863   
4  STORE_B  Electronics                 109       29084.83         9.944954   
5  STORE_B    Groceries                 115       28941.24        10.086957   

   max_revenue  
0       497.59  
1       499.74  
2       496.37  
3       495.24  
4       494.73  
5       497.43  

--- Multi-Function Dictionary Aggregation ---
         revenue                  units_sold           
             min   median     max        sum       mean
store_id                                               
STORE_A    20.21  260.040  499.74       3422   9.639437
STO

### Guided Question

When you execute `df.groupby('store_id')['revenue'].mean()`, the resulting Series has store_id set as its index.

What parameter can you pass directly into `.groupby()` to prevent the grouping key from becoming an index, and why is this useful when preparing features for machine learning models?

In [ ]:
mean_revenue = df.groupby("store_id", as_index=False)["revenue"].mean()

Pass `as_index=False` directly into `.groupby()`.

This keeps `store_id` as a regular column instead of making it the index.

Why this is useful for machine learning:
- Consistent DataFrame structure: The grouping key remains a normal column, making the result easier to manipulate.
- Feature preparation: You can merge aggregated features with other datasets using columns without needing to reset the index.
- Pipeline compatibility: Many preprocessing steps and ML workflows expect features to be stored in regular columns.

Key takeaway: `as_index=False` keeps grouping keys as columns, which often makes aggregated data easier to prepare for machine learning pipelines.


For example, compare the two approaches:

In [ ]:
# store_id becomes the index
df.groupby("store_id")["revenue"].mean()

# store_id remains a regular column
df.groupby("store_id", as_index=False)["revenue"].mean()

### Guided Question:

Suppose you want to normalize a numerical feature in your dataset by subtracting the mean of its respective category and dividing by the category's standard deviation (Z-score normalization per category).

Why would using `.agg()` fail to achieve this shape requirement, and how does `.transform()` solve the problem by broadcasting metrics back to the original rows?

`.agg()` fails to meet this shape requirement because it reduces each group to summary statistics, producing fewer rows than the original dataset. `.transform()` solves this by broadcasting each group's calculated mean and standard deviation back to every row belonging to that group, preserving the original DataFrame's shape.

1. Using `.agg()` — reduces the data

Suppose you have this dataset:

In [ ]:
df = pd.DataFrame({
    "category": ["A", "A", "B", "B"],
    "revenue": [10, 20, 100, 200]
})

If you calculate the mean and standard deviation using `.agg()`:

In [ ]:
stats = df.groupby("category")["revenue"].agg(["mean", "std"])

You get one row per category — only 2 rows instead of 4. Those statistics cannot be directly subtracted from the original four revenue values without additional alignment or mapping.

2. Using `.transform()` — preserves the data


In [2]:
grouped = df.groupby("category")["revenue"]

df["category_mean"] = grouped.transform("mean")
df["category_std"] = grouped.transform("std")

df["revenue_zscore"] = (
    (df["revenue"] - df["category_mean"])
    / df["category_std"]
)

Conceptually, the transformation looks like this:

Category	Revenue	Category mean	Category std
A	        10	    15	            7.07
A	        20	    15	            7.07
B	        100	    150	            70.71
B	        200	    150	            70.71

Each row receives the statistics for its own category. The resulting Z-score indicates how far that transaction's revenue is from its category's mean, measured in standard deviations.

Key takeaway:
- `.agg()` — summarizes groups; output shape generally shrinks.
- `.transform()` — calculates group-level statistics and maps them back to the original rows; output length matches the original group input.
This makes `.transform()` particularly useful for feature engineering, where you need to create normalized features while keeping every original observation available for machine learning.

Note: std uses the sample standard deviation by default (`ddof=1`). If a category has only one row, its standard deviation is `NaN`, so you should handle that case before training a model.

### Guided Question:

When applying custom logic inside `.agg()` or `.filter()`, writing a custom Python lambda function (e.g., `.agg(lambda x: x.max() - x.min())`) is often significantly slower than using built-in string methods (e.g., `np.ptp` or vectorized calls).

Why do built-in Pandas/NumPy aggregation methods execute faster than Python-level lambda iterations over group objects?

Built-in Pandas and NumPy aggregation methods are often faster because they reduce the amount of work performed by the Python interpreter and use optimized, lower-level implementations to process data.

1. Python overhead versus optimized operations
Consider these two approaches:

In [ ]:
# Custom Python lambda
result = df.groupby("category")["revenue"].agg(
    lambda x: x.max() - x.min()
)

# Built-in aggregation methods
result = df.groupby("category")["revenue"].agg(
    lambda x: np.ptp(x)
)

Both use a lambda, so the second example does not fully eliminate Python-level overhead. A clearer comparison is:

In [ ]:
# Custom aggregation
result = df.groupby("category")["revenue"].agg(
    lambda x: x.max() - x.min()
)

# Built-in aggregation for supported statistics
result = df.groupby("category")["revenue"].agg(["min", "max"])

The built-in version computes the minimum and maximum using optimized aggregation routines, though it returns two statistics rather than their difference.

2. Why built-in operations are generally faster
- Lower interpreter overhead: Python lambdas execute Python code and may be called separately for each group. Built-in aggregations can use optimized routines without repeatedly invoking custom Python functions.
- Optimized implementations: Many Pandas and NumPy operations use compiled C, Cython, or other low-level implementations rather than processing each value through a Python loop.
- Efficient memory handling: Vectorized operations can process arrays efficiently, reducing repeated Python-level operations and intermediate objects.
- Specialized groupby algorithms: Pandas can use optimized grouping and aggregation paths for recognized operations such as `sum`, `mean`, `min`, and `max`.

3. What about .filter()?

In [ ]:
# Custom Python function evaluated for each group
result = df.groupby("category").filter(
    lambda group: group["revenue"].sum() > 1000
)

Here, the lambda determines which groups to retain. The `sum()` itself is optimized, but the filtering condition still requires Python-level function execution for each group. For many groups, that overhead can become significant.

Key takeaway: Prefer built-in Pandas/NumPy operations when they express the required logic. However, not every lambda is slow, and not every vectorized operation is faster in every situation. Benchmark with representative data when performance matters.